# Anonymization Techniques: Applying Anonymization and Pseudonymization Methods

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply anonymization techniques
- Apply pseudonymization methods
- Protect personal data
- Understand k-anonymity
- Implement privacy-preserving methods

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 2, lesson 02 "Missing Values & Duplicates" — the record-matching skill that finds the same person twice is the skill that re-identifies them.

---

This notebook covers practical activities from **Course 06, Unit 3**:
- Anonymization Techniques: Applying anonymization and pseudonymization methods

---

## Introduction

**Anonymization and pseudonymization** protect personal data by removing or replacing identifying information, enabling data use while preserving privacy.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- The **real Titanic passenger manifest** (`Course 04/datasets/raw/titanic.csv`,
  891 real people). It has real direct identifiers (Name, Ticket, Cabin) and real
  quasi-identifiers (Sex, Pclass, Embarked, Age) - including 177 genuinely missing
  ages, which anonymization has to cope with.
- `pandas` and `hashlib`.

**Outputs:** What you'll see when you run the cells

- Pseudonymized records (real names replaced by salted-hash tokens).
- The **measured** k-anonymity of the real table, before and after generalization.
- The cost of reaching a target k: how many real passengers must be suppressed,
  and which people they turn out to be.

---


In [1]:
# Concept map: the two ways to de-identify data and the techniques behind them.
# Why first: the hands-on cell below applies these terms - read them here once.

import pandas as pd
import numpy as np
import hashlib

print("✅ Libraries imported!")
print("\nAnonymization and Pseudonymization")
print("=" * 60)

# Anonymization is IRREVERSIBLE: once done properly, no one can link the data
# back to a person - which is why GDPR no longer applies to truly anonymous data.
print("\nAnonymization:")
print("  - Remove identifiers")
print("  - Generalize data")
print("  - Suppress values")
print("  - k-anonymity")

# Pseudonymization is REVERSIBLE by whoever holds the mapping/salt -
# the data is still personal data under GDPR.
print("\nPseudonymization:")
print("  - Replace identifiers")
print("  - Reversible mapping")
print("  - Hash functions")
print("  - Tokenization")

print("\nTechniques:")
print("  - Generalization")
print("  - Suppression")
print("  - Perturbation")
print("  - Data masking")

print("\n✅ Anonymization concepts understood!")

✅ Libraries imported!

Anonymization and Pseudonymization

Anonymization:
  - Remove identifiers
  - Generalize data
  - Suppress values
  - k-anonymity

Pseudonymization:
  - Replace identifiers
  - Reversible mapping
  - Hash functions
  - Tokenization

Techniques:
  - Generalization
  - Suppression
  - Perturbation
  - Data masking

✅ Anonymization concepts understood!


## Hands-on: Anonymization, Pseudonymization, and k-Anonymity

**k-anonymity**: a table is *k*-anonymous if every combination of **quasi-identifiers**
(attributes that identify someone when combined - here sex, class, port, age) appears in at
least *k* rows. If a combination is unique (k = 1), that person can be singled out.

We measure k on the **real** 891-passenger manifest, raise it by **generalizing** the
quasi-identifiers, then **suppress** the rows that are still too rare - and finally count
what that protection cost us. Exercise 2 (Task 1) asks you to implement the same recipe.


In [2]:
# WHY on real data: k-anonymity is a property of a real population. Invented
# tables give whatever k you designed into them; a real manifest tells you how
# exposed real people actually are - and what protecting them costs.

# Practice: pseudonymize, then measure and improve k-anonymity

df = pd.read_csv('../../../Course 04/datasets/raw/titanic.csv')  # Load: 891 real passengers
print(f"Real Titanic manifest: {len(df)} passengers, {df.shape[1]} columns")
print(f"Direct identifiers present: Name, Ticket, Cabin")
print(f"Genuinely missing values: Age {int(df['Age'].isna().sum())}, "
      f"Cabin {int(df['Cabin'].isna().sum())}, Embarked {int(df['Embarked'].isna().sum())}")
print("\nFirst rows as they come out of the file:")
print(df[['Name', 'Sex', 'Age', 'Pclass', 'Embarked', 'Ticket']].head(5).to_string(index=False))

# --- Pseudonymize the direct identifiers ---
# A salted hash gives every person a stable token: the same name always maps to
# the same token, so records stay linkable without showing who they belong to.
SALT = 'unit3-demo'  # In production: a secret kept apart from the data
def token(value, salt=SALT):
    return hashlib.sha256((salt + str(value)).encode()).hexdigest()[:10]

df_pseudo = df.copy()
df_pseudo['person_id'] = df_pseudo['Name'].apply(token)  # Tokenize: real name -> pseudonym
df_pseudo = df_pseudo.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])
df_pseudo = df_pseudo[['person_id', 'Sex', 'Age', 'Pclass', 'Embarked', 'Survived']]
print("\nAfter pseudonymization (name -> salted hash, ticket and cabin dropped):")
print(df_pseudo.head(5).to_string(index=False))

# --- Measure k-anonymity over the quasi-identifiers ---
def k_anonymity(table, quasi_identifiers):
    """Smallest group size over the quasi-identifier combinations that occur.
    dropna=False keeps the missing-age rows as their own group - they are real
    people too, and ignoring them would flatter the result."""
    return int(table.groupby(quasi_identifiers, observed=True, dropna=False).size().min())

QI = ['Sex', 'Pclass', 'Embarked', 'Age']
k_before = k_anonymity(df_pseudo, QI)
sizes_before = df_pseudo.groupby(QI, observed=True, dropna=False).size()
print(f"\nk-anonymity over {QI}: k = {k_before}")
print(f"{int((sizes_before == 1).sum())} of the {len(sizes_before)} combinations that occur "
      f"are held by exactly ONE passenger.")
print("k = 1 means those people have a UNIQUE quasi-identifier combination - an")
print("attacker who knows their sex, class, port and age can single them out,")
print("even though we deleted every name.")

# --- Step A: Generalize quasi-identifiers (coarser values -> bigger groups) ---
AGE_BINS, AGE_LABELS = [0, 18, 30, 45, 120], ['<=18', '19-30', '31-45', '46+']
df_gen = df_pseudo.copy()
df_gen['Age'] = pd.cut(df_gen['Age'], bins=AGE_BINS, labels=AGE_LABELS)  # Generalize: exact age -> band
print("\nAfter generalization (exact age -> 4 age bands):")
print(df_gen.head(5).to_string(index=False))
k_gen = k_anonymity(df_gen, QI)
sizes_gen = df_gen.groupby(QI, observed=True, dropna=False).size()
print(f"k-anonymity after generalization: k = {k_gen}  (was {k_before})")
print(f"Combinations held by a single passenger: "
      f"{int((sizes_gen == 1).sum())} (was {int((sizes_before == 1).sum())})")
print("-> Generalization removed most of the exposure, but k is still 1: a few")
print("   real passengers remain unique even inside broad age bands.")

# --- Step B: Suppress rows still in groups smaller than the target k ---
TARGET_K = 5  # Common regulatory rule of thumb for released microdata
group_sizes = df_gen.groupby(QI, observed=True, dropna=False)['person_id'].transform('size')
df_k = df_gen[group_sizes >= TARGET_K]  # Keep: rows whose group is big enough
suppressed = df_gen[group_sizes < TARGET_K]  # Drop: the still-identifiable rows
k_after = k_anonymity(df_k, QI)
print(f"\nSuppressing every row whose group is smaller than k={TARGET_K}:")
print(f"  released: {len(df_k)} passengers    suppressed: {len(suppressed)} "
      f"({len(suppressed)/len(df_gen):.1%} of the manifest)")
print(f"  final k-anonymity: k = {k_after} (target was {TARGET_K})")
print("Generalize first, then suppress the stubborn outliers - the standard recipe.")

# --- What did the protection cost? Measure it, do not assume it ---
# The suppressed rows are the RAREST people, so releasing only the survivors
# changes the picture the data gives - which is the privacy/utility trade-off.
print("\nCOST OF PROTECTION (real numbers, not a rule of thumb):")
print(f"  survival rate   before {df_gen['Survived'].mean():.1%}  ->  after {df_k['Survived'].mean():.1%}")
print(f"  share female    before {(df_gen['Sex'] == 'female').mean():.1%}  ->  after {(df_k['Sex'] == 'female').mean():.1%}")
print("\nRead the suppressed group before you accept the trade-off:")
print(suppressed.groupby(['Sex', 'Pclass'], observed=True).size().to_string())
# Say it with a number: are the suppressed people a random slice of the ship?
share_female_all = (df_gen['Sex'] == 'female').mean()
share_female_supp = (suppressed['Sex'] == 'female').mean()
print(f"\nWomen are {share_female_all:.1%} of the manifest but "
      f"{share_female_supp:.1%} of the {len(suppressed)} suppressed passengers.")

print("\n-> Suppression removes the people with rare attribute combinations, and")
print("   rare combinations are exactly where minorities live. Privacy protection")
print("   quietly reshapes who is represented in the released data - Unit 2's")
print("   fairness problem, created by Unit 3's privacy fix. Both matter; you have")
print("   to decide the balance openly, and document it.")
print("\n\u2705 You just did what Exercise 2 Task 1 asks: check k, generalize, re-check.")


Real Titanic manifest: 891 passengers, 12 columns
Direct identifiers present: Name, Ticket, Cabin
Genuinely missing values: Age 177, Cabin 687, Embarked 2

First rows as they come out of the file:
                                               Name    Sex  Age  Pclass Embarked           Ticket
                            Braund, Mr. Owen Harris   male 22.0       3        S        A/5 21171
Cumings, Mrs. John Bradley (Florence Briggs Thayer) female 38.0       1        C         PC 17599
                             Heikkinen, Miss. Laina female 26.0       3        S STON/O2. 3101282
       Futrelle, Mrs. Jacques Heath (Lily May Peel) female 35.0       1        S           113803
                           Allen, Mr. William Henry   male 35.0       3        S           373450

After pseudonymization (name -> salted hash, ticket and cabin dropped):
 person_id    Sex  Age  Pclass Embarked  Survived
088be50159   male 22.0       3        S         0
4d0f50b974 female 38.0       1        C   

---

## ➡️ You've Finished Unit 3

With notebooks 06-07 done, you have now *practiced* the two core protection techniques
from Notebook 01 (encryption; anonymization/pseudonymization) and measured k-anonymity.

**Next**: the unit exercises (`exercises/exercise_01.ipynb`, `exercises/exercise_02.ipynb`),
then **Unit 4: Transparency and Accountability** (`unit4-transparency-accountability/`).

## 📚 References

1. Sweeney, L. (2002). *k-Anonymity: A Model for Protecting Privacy*. International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems, 10(5).
2. Machanavajjhala, A., Kifer, D., Gehrke, J. & Venkitasubramaniam, M. (2007). *l-Diversity: Privacy Beyond k-Anonymity*. ACM Transactions on Knowledge Discovery from Data, 1(1).
3. Narayanan, A. & Shmatikov, V. (2007). *How To Break Anonymity of the Netflix Prize Dataset*. <https://arxiv.org/abs/cs/0610105>
4. European Parliament & Council (2016). *Regulation (EU) 2016/679 — General Data Protection Regulation (GDPR)*. <https://gdpr-info.eu/>